# PSALM — Minimal-pair scoring

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SharathSPhD/PSALM/blob/main/notebooks/01_minimal_pairs.ipynb)

Load a published ELC-PSALM checkpoint from the Hugging Face Hub and score English
syntactic minimal pairs by pseudo-log-likelihood (Salazar et al., 2020). The model
should assign higher likelihood to the grammatical sentence in each pair.

This notebook is self-contained: it implements PLL directly on top of
`AutoModelForMaskedLM`, so it works on the free Colab CPU/GPU runtime with no repo
checkout required.

In [ ]:
%pip install -q --upgrade transformers torch sentencepiece

In [ ]:
import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer

# Published ablation arm (English-dose control). Swap for psalm-arm-b/-c/-d to
# compare doses, or psalm-submission for the leaderboard-track model.
MODEL_ID = "qbz506/psalm-arm-a"

device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForMaskedLM.from_pretrained(MODEL_ID, trust_remote_code=True).to(device).eval()
print("loaded", MODEL_ID, "on", device)

In [ ]:
@torch.no_grad()
def pseudo_log_likelihood(text: str) -> float:
    """Sum of log p(token | rest) with each token masked in turn."""
    ids = tok(text, return_tensors="pt").input_ids.to(device)
    mask_id = tok.mask_token_id if tok.mask_token_id is not None else tok.vocab_size - 1
    total = 0.0
    for i in range(ids.size(1)):
        masked = ids.clone()
        gold = ids[0, i].item()
        masked[0, i] = mask_id
        logits = model(masked).logits[0, i]
        total += torch.log_softmax(logits, dim=-1)[gold].item()
    return total


pairs = [
    ("the cat sleeps on the mat", "the cat sleep on the mat"),
    ("she has finished her work", "she have finished her work"),
    ("the books are on the table", "the books is on the table"),
    ("every student read the book", "every students read the book"),
]
correct = 0
for good, bad in pairs:
    sg, sb = pseudo_log_likelihood(good), pseudo_log_likelihood(bad)
    ok = sg > sb
    correct += ok
    print(f"[{'OK ' if ok else 'MISS'}] {sg:8.2f} > {sb:8.2f}  | {good!r} vs {bad!r}")
print(f"\naccuracy on {len(pairs)} demo pairs: {correct / len(pairs):.2f}")

The full BLiMP suite (thousands of pairs across dozens of paradigms) is run by
`scripts/official_eval.py` in the repository; this demo uses a handful of pairs to
show the mechanism. See the [results page](https://SharathSPhD.github.io/PSALM/results/)
for the full, seed-replicated numbers.